# Update meta data and objects 

In [ ]:
import scanpy as sc

import numpy as np
import pandas as pd

import os

In [ ]:
# rpy2 
os.environ['R_HOME'] = '/nobackup/peer/fdeckert/miniconda3/envs/r.4.1.0-FD20220623HSCT/lib/R'

In [ ]:
sc.set_figure_params(figsize=(10, 10), dpi_save=1200, fontsize=18, frameon=False)

In [ ]:
os.chdir('/research/peer/fdeckert/FD20220623HSCT')

In [ ]:
# Load color 
import rpy2.robjects as robjects
color_load = robjects.r.source('plotting_global.R')
color = dict()
for i in range(len(color_load[0])):
    color[color_load[0].names[i]] = {key : color_load[0][i].rx2(key)[0] for key in color_load[0][i].names}

In [ ]:
# Set color function 
def set_color(categories, adata): 
    
    categories = [x for x in categories if x in list(adata.obs.columns)]
    
    for category in categories: 
        
        adata.obs[category] = pd.Series(adata.obs[category], dtype='category').cat.remove_unused_categories()
        
        keys = list(color[category].keys())
        keys = [x for x in keys if x in list(adata.obs[category])]        

        adata.obs[category] = adata.obs[category].cat.reorder_categories(keys)
        adata.uns[category+'_colors'] = np.array([color[category].get(key) for key in keys], dtype=object)
        
    return adata

# Files and parameter 

In [ ]:
# Seurat object 
pp_h5ad_file = 'data/object/pp.h5ad'
pp_subset_h5ad_file = 'data/object/pp_subset.h5ad'

# Meta data 
cell_type_leiden_subset_file_1 = 'result/annotation/sbusets/lymphocyte_p.csv'
cell_type_leiden_subset_file_2 = 'result/annotation/sbusets/myeloid_p.csv'
cell_type_leiden_subset_file_3 = 'result/annotation/sbusets/structural_p.csv'

# UMAP 
umap_file = 'data/object/subsets/scvi/components/umap.csv'

# Import 

In [ ]:
adata = sc.read_h5ad(pp_h5ad_file)

In [ ]:
cell_type_leiden_subset_1 = pd.read_csv(cell_type_leiden_subset_file_1, index_col=0).assign(domain_leiden_subset='lymphocyte')
cell_type_leiden_subset_2 = pd.read_csv(cell_type_leiden_subset_file_2, index_col=0).assign(domain_leiden_subset='myeloid')
cell_type_leiden_subset_3 = pd.read_csv(cell_type_leiden_subset_file_3, index_col=0).assign(domain_leiden_subset='structural')

In [ ]:
cell_type_leiden_subset = pd.concat([cell_type_leiden_subset_1, cell_type_leiden_subset_2, cell_type_leiden_subset_3])

In [ ]:
umap = pd.read_csv(umap_file, index_col=0).to_numpy()

# Meta data 

In [ ]:
adata.obs = adata.obs.merge(cell_type_leiden_subset, left_index=True, right_index=True, how='left')

# Subset 

In [ ]:
adata = adata[pd.isna(adata.obs['domain_leiden_subset'])==False]

# UMAP

In [ ]:
adata.obsm['X_umap'] = umap

# Color 

In [ ]:
# Set colors
adata = set_color(list(color.keys()), adata)

# UMAP

In [ ]:
sc.pl.umap(adata, color=['cell_type_leiden', 'cell_type_leiden_subset', 'patient_id', 'hto_demux_class', 'time_point', 'cc_phase_class', 'digestion_protocol', 'genotype_class', 'cc_aa_alignment_size'], size=20, ncols=4, wspace=0.4)

In [ ]:
sc.pl.umap(adata, color=['celltypist_immun_high'], size=20, ncols=1, wspace=0.4)

In [ ]:
sc.pl.umap(adata, color=['celltypist_immun_low'], size=20, ncols=1, wspace=0.4)

# Save result 

In [ ]:
adata.write_h5ad(pp_subset_h5ad_file)